# Project 09: Permutation Importance vs Built-in Importance
## An Adult Income study
**Name:** B.Vishnutharan  
**Registration number:** 2021/ICT/106

**Project question:** How do Random Forest's built-in feature importance and permutation importance differ when predicting adult income, and when can their rankings be misleading?

I chose this question before loading the data. The aim is to check how much trust to place in a model's feature rankings.

Run the cells from top to bottom. The saved outputs show data loading and cleaning checks for this stage. Cleaned data is in `data/`, and audit results are in `results/`. Later commits add models, importance comparisons, and error reviews.

## 1. Plan and data choice
I chose the public UCI Adult Income dataset because it contains both numbers and categories. It has 48,842 rows and 14 input columns before cleaning. The target is whether a person's annual income was above $50,000. The data comes from the 1994 US Census, so it does not describe present-day income in Sri Lanka.

The work follows these steps:
1. Check and clean the data.
2. Keep the original test file separate.
3. Compare a majority-class baseline, Logistic Regression, and Random Forest using training data.
4. Select Random Forest settings using cross-validation.
5. Measure final results on the test set.
6. Compare importance methods and run a separate random-column experiment.
7. Review at least 10 wrong predictions and explain the limits.

**Source:** [UCI Adult dataset](https://archive.ics.uci.edu/dataset/2/adult). Becker, B. and Kohavi, R. (1996), DOI: 10.24432/C5XW20. License: CC BY 4.0.

## 2. Set up the project
The seed fixes random choices so the results can be repeated. A pipeline keeps data preparation and model fitting together. Each validation fold learns its own preparation steps from its training rows.

The notebook works from the project folder and uses local data files. It does not need an online account or an API key.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_validate, GroupShuffleSplit
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                            recall_score, f1_score, roc_auc_score, average_precision_score,
                            confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
                            PrecisionRecallDisplay, classification_report)

ROOT = Path.cwd()
assert (ROOT / 'data/raw/adult.data').exists(), 'Open this notebook from the project folder.'
for folder in ['results', 'figures']:
    (ROOT / folder).mkdir(exist_ok=True)
SEED = 106
REPEATS = 10
PERM_ROWS = 2500
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'savefig.bbox': 'tight'})

def save_chart(name):
    plt.savefig(ROOT / 'figures' / f'{name}.png', dpi=160)
    plt.show()
    plt.close()

print('Python:', platform.python_version())
print('Random seed:', SEED)

Python: 3.13.2
Random seed: 106


## 3. Load the original files
`adult.data` is the supplied training file. `adult.test` is the supplied test file. A question mark means a missing value. Some test labels end with a full stop, which needs to be removed.

The file checks below help confirm that the same source files are used in another run.

In [2]:
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
           'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
           'hours-per-week', 'native-country', 'income']
features = columns[:-1]
numeric = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical = [c for c in features if c not in numeric]

def read_adult(filename):
    frame = pd.read_csv(ROOT / 'data/raw' / filename, names=columns,
                        skipinitialspace=True, na_values='?', comment='|')
    for col in frame.select_dtypes(include=['object', 'string']).columns:
        frame[col] = frame[col].str.strip()
    frame['income'] = frame['income'].str.rstrip('.')
    assert set(frame['income'].dropna()) == {'<=50K', '>50K'}
    return frame

raw_train = read_adult('adult.data')
raw_test = read_adult('adult.test')
checksums = {name: hashlib.sha256((ROOT / 'data/raw' / name).read_bytes()).hexdigest()
             for name in ['adult.data', 'adult.test', 'adult.names']}
(ROOT / 'results/source_checksums.json').write_text(json.dumps(checksums, indent=2))
print('Original training rows:', len(raw_train))
print('Original test rows:', len(raw_test))
assert len(raw_train) + len(raw_test) == 48842
display(raw_train.head())

Original training rows: 32561
Original test rows: 16281


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 4. Data checks and repeated records
I checked missing values, repeated rows, and repeated input profiles because these can affect the result.

An input profile means all 14 input values together. Exact repeated rows are kept once in each file. Any test profile already present in training is removed from the test set. This makes the test stricter, but it also means the final score applies to a cleaned test set rather than the full original test file.

If two training rows have the same input profile, they stay in the same cross-validation fold. This also covers the rare case where the same inputs have different income labels.

In [3]:
train = raw_train.drop_duplicates().copy()
test_unique = raw_test.drop_duplicates().copy()
train_profiles = pd.util.hash_pandas_object(train[features], index=False)
test_profiles = pd.util.hash_pandas_object(test_unique[features], index=False)
overlap = test_profiles.isin(set(train_profiles))
test = test_unique.loc[~overlap].copy()

audit = {'raw_train_rows': len(raw_train), 'raw_test_rows': len(raw_test),
         'train_exact_duplicates_removed': len(raw_train) - len(train),
         'test_exact_duplicates_removed': len(raw_test) - len(test_unique),
         'overlapping_test_profiles_removed': int(overlap.sum()),
         'clean_train_rows': len(train), 'clean_test_rows': len(test)}
display(pd.Series(audit, name='Rows').to_frame())
missing = train[features].isna().sum().sort_values(ascending=False)
display(missing.rename('Missing training values').to_frame())
assert not set(pd.util.hash_pandas_object(test[features], index=False)) & set(train_profiles)
assert train[numeric].notna().all().all()
assert train['age'].between(17, 100).all()
assert train['hours-per-week'].between(1, 99).all()

train.to_csv(ROOT / 'data/adult_train_clean.csv', index=False)
test.to_csv(ROOT / 'data/adult_test_clean.csv', index=False)
pd.Series(audit).to_csv(ROOT / 'results/data_audit.csv', header=['rows'])
X_train, y_train = train[features].copy(), train['income'].eq('>50K').astype(int)
X_test, y_test = test[features].copy(), test['income'].eq('>50K').astype(int)
groups = pd.util.hash_pandas_object(X_train, index=False).to_numpy()

,Rows
raw_train_rows,32561
raw_test_rows,16281
train_exact_duplicates_removed,24
test_exact_duplicates_removed,5
overlapping_test_profiles_removed,26
clean_train_rows,32537
clean_test_rows,16250


,Missing training values
occupation,1843
workclass,1836
native-country,582
education,0
fnlwgt,0
age,0
marital-status,0
education-num,0
race,0
relationship,0
